### **Validate configs**

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents)
    if (path / "pyproject.toml").exists()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Wrong location.")

load_dotenv(PROJECT_ROOT/".env", override=True)

required_variables = ["OPENAI_API_KEY",
"LLM_MODEL",
"AZURE_OPENAI_API_KEY",
"AZURE_OPENAI_BASE_URL",
"AZURE_OPENAI_DEPLOYMENT"]

missing = [
    name for name in required_variables
    if not os.getenv(name, "").strip()
]

if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

azure_base_url = os.environ["AZURE_OPENAI_BASE_URL"].strip().rstrip("/") + "/"

if not azure_base_url.endswith("/openai/v1/"):
    raise ValueError(
        "AZURE_OPENAI_BASE_URL must be the Azure OpenAI v1 URL "
        "ending in /openai/v1/."
    )

print("Configuration loaded. All required values are present.")

Configuration loaded. All required values are present.


In [2]:
openai_model_name = os.getenv("LLM_MODEL")

openai_llm = ChatOpenAI(
    model=openai_model_name,
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://api.openai.com/v1",
    max_tokens=128,
    timeout=60,
    max_retries=0,
)

azure_llm = ChatOpenAI(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=azure_base_url,
    reasoning_effort="minimal",
    max_completion_tokens=1024,
    timeout=60,
    max_retries=0,
)

print("Both clients created. No model calls made yet.")

Both clients created. No model calls made yet.


#### ***Connection Test***

In [3]:
responses = {}

def test_provider(provider_name, model):
    try:
        response = model.invoke(
            "Reply with exactly: Aegis connection successful."
        )
    except Exception as exc:
        # Avoid printing raw exception messages or request details.
        print(f"{provider_name}: FAILED")
        print(f"Error type: {type(exc).__name__}")

        status_code = getattr(exc, "status_code", None)
        if status_code is not None:
            print(f"HTTP status: {status_code}")

        return None

    responses[provider_name] = response

    print(f"Provider: {provider_name}")
    print(f"Response: {response.content}")
    print(f"Token usage: {response.usage_metadata}")
    print(f"Finish reason: {response.response_metadata.get('finish_reason')}")

    return response

In [4]:
openai_response = test_provider("openai", openai_llm)

Provider: openai
Response: Aegis connection successful.
Token usage: {'input_tokens': 16, 'output_tokens': 9, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Finish reason: stop


In [5]:
azure_response = test_provider("azure", azure_llm)

Provider: azure
Response: Aegis connection successful.
Token usage: {'input_tokens': 16, 'output_tokens': 15, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Finish reason: stop


In [6]:
from aegis.config import get_settings
from aegis.llm.gateway import get_model
from aegis.llm.usage import extract_usage

settings = get_settings()

# Nodes ask for a role; settings decide the provider and model (#11).
role = "researcher"
role_config = settings.role_models[role]
model = get_model(role)

try:
    response = model.invoke("Reply with exactly: Aegis connection successful.")
except Exception as exc:
    print(f"Request failed: {type(exc).__name__}")
else:
    print(response.content)
    print(
        extract_usage(
            response,
            provider=role_config.provider,
            requested_model=role_config.model,
        )
    )

Aegis connection successful.
{'provider': 'azure', 'requested_model': 'gpt-5-mini', 'reported_model': 'gpt-5-mini-2025-08-07', 'input_tokens': 16, 'output_tokens': 15, 'total_tokens': 31, 'cached_input_tokens': 0, 'reasoning_tokens': 0}


## Cost calculation

Convert token usage into an estimated USD token cost using the
configured price table.

Cached input is a subset of total input.
Reasoning tokens are already included in output tokens.

In [7]:
from decimal import Decimal

from aegis.llm.pricing import calculate_cost


# Synthetic usage: this cell does not call a provider.
usage = {
    "provider": "openai",
    "requested_model": "gpt-5.4",
    "input_tokens": 1000,
    "cached_input_tokens": 0,
    "output_tokens": 500,
    "reasoning_tokens": 200,
}

cost = calculate_cost(usage)

# Independently calculated using the configured rates:
# Input:  1000 × $2.50 / 1,000,000 = $0.0025
# Output:  500 × $15.00 / 1,000,000 = $0.0075
# Total:                             $0.0100
assert cost == Decimal("0.0100")

print("Estimated token cost, USD:", cost)
print("PASS: cost matches the manual calculation")

Estimated token cost, USD: 0.01
PASS: cost matches the manual calculation


In [8]:
# Now assume 500 of the 1,000 input tokens were served from cache.
cached_usage = {
    **usage,
    "cached_input_tokens": 500,
}

cached_cost = calculate_cost(cached_usage)

# Uncached input: 500 × $2.50 / 1M = $0.00125
# Cached input:   500 × $0.25 / 1M = $0.000125
# Output:         500 × $15.00 / 1M = $0.0075
# Total:                             $0.008875
assert cached_cost == Decimal("0.008875")
assert cached_cost < cost

print("Cost with cached input, USD:", cached_cost)
print("PASS: cached tokens use the lower configured rate")

Cost with cached input, USD: 0.008875
PASS: cached tokens use the lower configured rate


## Model by role (#11)

Each role maps to a provider and model in settings. Override one field from `.env`
with nested variables, e.g. `ROLE_MODELS__JUDGE__PROVIDER=azure`.

In [9]:
for role, config in settings.role_models.items():
    print(
        f"{role:<10} {config.provider:<6} {config.model:<11} "
        f"max_completion_tokens={config.max_completion_tokens:<5} "
        f"reasoning_effort={config.reasoning_effort}"
    )

# The verifier should not mark its own provider's work.
assert get_model("judge").metadata["aegis_provider"] != get_model("researcher").metadata["aegis_provider"]
print("\nPASS: judge and researcher use different providers")

planner    openai gpt-5.4     max_completion_tokens=4096  reasoning_effort=low
researcher azure  gpt-5-mini  max_completion_tokens=2048  reasoning_effort=minimal
analyst    azure  gpt-5-mini  max_completion_tokens=2048  reasoning_effort=low
judge      openai gpt-5.4     max_completion_tokens=2048  reasoning_effort=low
reporter   openai gpt-5.4     max_completion_tokens=4096  reasoning_effort=low

PASS: judge and researcher use different providers


## Automatic usage tracking (#12)

`new_run_budget()` returns a tracker and a `callbacks` config. Passing the config to
`invoke` checks the budget before the call and records its cost after, with no extra code.

In [10]:
from aegis.llm.callbacks import new_run_budget

tracker, config = new_run_budget(settings)

for role in ["researcher", "planner"]:
    get_model(role).invoke("Reply with one word: ready.", config=config)

for call in tracker.calls:
    print(call)

print(f"\nSpent: ${tracker.spent_usd:.6f} of ${tracker.limit_usd}")
assert len(tracker.calls) == 2

CallRecord(provider='azure', model='gpt-5-mini', input_tokens=13, output_tokens=10, cached_input_tokens=0, reasoning_tokens=0, cost_usd=Decimal('0.00002325'))
CallRecord(provider='openai', model='gpt-5.4', input_tokens=13, output_tokens=4, cached_input_tokens=0, reasoning_tokens=0, cost_usd=Decimal('0.0000925'))

Spent: $0.000116 of $1.00


### Budget cutoff (#10 "Done when")

A tracker with a $0.0001 cap allows the first live call, then blocks the second.

In [11]:
from aegis.llm.budget import BudgetExceededError, BudgetTracker
from aegis.llm.callbacks import BudgetCallbackHandler

tiny_tracker = BudgetTracker("0.0001")
tiny_config = {"callbacks": [BudgetCallbackHandler(tiny_tracker)]}
researcher = get_model("researcher")

# A one-word reply costs about $0.00002, so ask for a longer answer
# to push the first call over the $0.0001 cap.
researcher.invoke("List the numbers 1 to 60, separated by commas.", config=tiny_config)
print(f"First call cost: ${tiny_tracker.calls[0].cost_usd:.6f}")

try:
    researcher.invoke("Reply with one word: second.", config=tiny_config)
except BudgetExceededError as exc:
    print(f"Second call blocked: {exc}")
else:
    raise AssertionError("The second call should have been blocked")

assert len(tiny_tracker.calls) == 1

budget_exhausted spent_usd=0.000379 limit_usd=0.0001; further calls will be blocked


Error in BudgetCallbackHandler.on_chat_model_start callback: BudgetExceededError('Budget of $0.0001 used up (spent $0.000379); call blocked.')


First call cost: $0.000379
Second call blocked: Budget of $0.0001 used up (spent $0.000379); call blocked.


## Bad-credential handling (#13)

Each misconfiguration fails before a model runs, so these calls cost nothing.
The gateway raises an Aegis error that keeps the provider, model and status,
and drops the raw provider text (OpenAI's 401 message echoes part of the key).

In [12]:
from pydantic import SecretStr

from aegis.config import RoleModel
from aegis.llm import gateway
from aegis.llm.errors import ProviderError

real_get_settings = gateway.get_settings

researcher_config = settings.role_models["researcher"]
misconfigurations = {
    "wrong OpenAI key": (
        "planner",
        {"openai_api_key": SecretStr("sk-aegis-invalid-key-for-notebook-0000")},
    ),
    "wrong Foundry deployment": (
        "researcher",
        {
            "role_models": {
                **settings.role_models,
                "researcher": RoleModel(
                    **{**researcher_config.model_dump(), "model": "aegis-missing-deployment"}
                ),
            }
        },
    ),
    "unreachable base URL": (
        "researcher",
        {"azure_openai_base_url": "https://aegis-unreachable.invalid/openai/v1/"},
    ),
}

try:
    for name, (role, update) in misconfigurations.items():
        # model_copy skips validation, so the broken value gets through.
        broken = settings.model_copy(update=update)
        gateway.get_settings = lambda broken=broken: broken
        try:
            gateway.get_model(role).invoke("hello")
        except ProviderError as exc:
            print(f"{name:<26} -> {type(exc).__name__}: {exc}")
        else:
            raise AssertionError(f"{name} should have failed")
finally:
    gateway.get_settings = real_get_settings

wrong OpenAI key           -> ProviderAuthError: authentication failed: check the API key (provider=openai, model=gpt-5.4, status=401)


wrong Foundry deployment   -> ProviderNotFoundError: model or deployment not found: check its name and the base URL (provider=azure, model=aegis-missing-deployment, status=404)
unreachable base URL       -> ProviderUnavailableError: provider unreachable or unavailable (provider=azure, model=gpt-5-mini)
